# Risk Stratified Table

**This notebook generates a Table 1 for patients with high PD-L1 advanced non-small cell lung cancer receiving first-line pembrolizumab plus chemotherapy versus pembrolizumab, showing covariate differences stratified by the threshold effect.** 

In [1]:
import numpy as np
import pandas as pd

from tableone import TableOne

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
14967,F15192E35AE2D,pembro,2021-09-01
12460,FC5DD0D8E5BD6,pembro,2021-05-19
13863,F56A529962748,pembro,2017-11-30


In [4]:
treatment_df.shape

(15318, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro_platinum').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(1434, 164)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,FF16F972863F8,0.512073
1,F95B93B796545,0.923535
2,FD0C44C783F30,0.411457


In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(1434, 167)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(1434, 168)

In [40]:
#for ASCO abstract
from lifelines import KaplanMeierFitter

# Reverse KM: treat deaths as censored, censoring as events
kmf = KaplanMeierFitter()
kmf.fit(durations=df['duration'], event_observed=(df['event'] == 0)) 
median_followup = kmf.median_survival_time_

print(f'median age: {df.age.median()}')
print(f'percent male: {df.sex_male.sum()/len(df)}')
print(f'median follow-up: {median_followup/30}')

median age: 71.0
percent male: 0.5251046025104602
median follow-up: 37.96666666666667


## Binning relevant covariates

In [14]:
df['weight_loss_5p'] = np.where(df['percent_change_weight'] <= -5, 1, 0)

In [15]:
df['creatinine_2'] = np.where(df['creatinine'] > 2, 1, 0)

In [16]:
df['hemoglobin_9'] = np.where(df['hemoglobin'] < 9, 1, 0)

In [17]:
df['ast_60'] = np.where(df['ast'] > 60, 1, 0)

In [18]:
df['albumin_3'] = np.where(df['albumin'] < 30, 1, 0)

In [19]:
df['alp_200'] = np.where(df['alp'] > 200, 1, 0)

In [20]:
df['above_threshold'] = np.where(df['psurv_180_calibrated'] >= 0.63, 1, 0)

## Creating table 

In [21]:
table1 = TableOne(
    data = df, 
    columns = ['age', 'GroupStage_mod', 'Histology', 'SmokingStatus', 'ecog_index', 'weight_loss_5p', 'KRAS_status', 'BRAF_status', 'MET_status', 'RET_status', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'liver_met', 'brain_met', 'bone_met', 'adrenal_met'],
    categorical = ['GroupStage_mod', 'Histology', 'SmokingStatus', 'ecog_index', 'weight_loss_5p', 'KRAS_status', 'BRAF_status', 'MET_status', 'RET_status', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'liver_met', 'brain_met', 'bone_met', 'adrenal_met'],
    continuous = ['age'],
    nonnormal = ['age'],
    groupby = 'above_threshold',
    pval = True)

In [22]:
table1

Grouped by above_threshold                                                              
                                                                     Missing           Overall                 0                 1 P-Value
n                                                                                         1434               454               980        
age, median [Q1,Q3]                                                        0  71.0 [64.0,77.8]  73.0 [65.0,81.0]  70.0 [63.0,76.0]  <0.001
GroupStage_mod, n (%) 1.0                                                            110 (7.7)          27 (5.9)          83 (8.5)   0.015
                      2.0                                                             68 (4.7)          16 (3.5)          52 (5.3)        
                      3.0                                                           167 (11.6)          42 (9.3)        125 (12.8)        
                      4.0                                                          1089 (75.9)        369 (81.3)        720 (73.5)        
Histology, n (%)      NSCLC histology NOS                                             96 (6.7)          37 (8.1)          59 (6.0)  <0.001
                      Non-squamous cell carcinoma                                  1022 (71.3)        286 (63.0)        736 (75.1)        
                      Squamous cell carcinoma                                       316 (22.0)        131 (28.9)        185 (18.9)        
SmokingStatus, n (%)  History of smoking                                           1338 (93.3)        425 (93.6)        913 (93.2)   0.305
                      No history of smoking                                           95 (6.6)          28 (6.2)          67 (6.8)        
                      Unknown/Not documented                                           1 (0.1)           1 (0.2)           0 (0.0)        
ecog_index, n (%)     0.0                                                           310 (21.6)          41 (9.0)        269 (27.4)  <0.001
                      1.0                                                           849 (59.2)        247 (54.4)        602 (61.4)        
                      2.0                                                           232 (16.2)        135 (29.7)          97 (9.9)        
                      3.0                                                             40 (2.8)          28 (6.2)          12 (1.2)        
                      4.0                                                              3 (0.2)           3 (0.7)           0 (0.0)        
weight_loss_5p, n (%) 0                                                            1144 (79.8)        287 (63.2)        857 (87.4)  <0.001
                      1                                                             290 (20.2)        167 (36.8)        123 (12.6)        
KRAS_status, n (%)    None                                                          479 (33.4)        164 (36.1)        315 (32.1)   0.192
                      negative                                                      506 (35.3)        147 (32.4)        359 (36.6)        
                      positive                                                      421 (29.4)        131 (28.9)        290 (29.6)        
                      unknown                                                         28 (2.0)          12 (2.6)          16 (1.6)        
BRAF_status, n (%)    None                                                          423 (29.5)        156 (34.4)        267 (27.2)   0.035
                      negative                                                      913 (63.7)        268 (59.0)        645 (65.8)        
                      positive                                                        72 (5.0)          20 (4.4)          52 (5.3)        
                      unknown                                                         26 (1.8)          10 (2.2)          16 (1.6)        
MET_status, n (%)  

In [23]:
table1.to_csv('../outputs/risk_stratified_table1.csv', header = True)

In [24]:
df.groupby('above_threshold')['ecog_index_na'].sum()

above_threshold
0    115
1    248
Name: ecog_index_na, dtype: int64